### Enrichr links:

SBP_pre pooled : https://maayanlab.cloud/Enrichr/enrich?dataset=c845e2082cbdb16705b777f6aece2af1

SBP_pre_1 : https://maayanlab.cloud/Enrichr/enrich?dataset=5bf9a6c519670363df98063f3e7ed44b

SBP_pre_2 : https://maayanlab.cloud/Enrichr/enrich?dataset=492ff1aa34c716b292802953d014b58b

DBP_pre pooled : https://maayanlab.cloud/Enrichr/enrich?dataset=a7b0625069bfb3792d62016f42dd4803

DBP_pre_1: https://maayanlab.cloud/Enrichr/enrich?dataset=e999991be56702e5edefe6713ee7392a

DBP_pre_2: https://maayanlab.cloud/Enrichr/enrich?dataset=e35bbf0d929054425f0af65ec5bec6f5

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

ROOT = Path("<PATH_TO_ENRICHMENT_DIR>")

RESOURCE_FILES = {
    "<PATH_TO_ENCODE_CHEA_CHIPX_TFS_TXT>": "ENCODE",
    "<PATH_TO_GO_MOLECULAR_FUNCTION_2025_TXT>": "GO_MF",
    "<PATH_TO_REACTOME_PATHWAYS_2024_TXT>": "Reactome",
    "<PATH_TO_GWAS_CATALOG_2025_TXT>": "GWAS_Cat",
}

DESIRED_ORDER = [
    "SBP_pre_ADJ",
    "SBP_pre_1_ADJ",
    "SBP_pre_2_ADJ",
    "SBP_pre_4_ADJ",
    "DBP_pre_ADJ",
    "DBP_pre_1_ADJ",
    "DBP_pre_2_ADJ",
    "DBP_pre_4_ADJ",
]

KEYWORDS_RE = re.compile(
    r"(systolic|diastolic|medication|cardio|heart|myocardial|renin|electrocardiogram|Pr Interval|diuretics|infraction)",
    re.IGNORECASE,
)

def load_all_tables(root):
    frames = []
    for pheno_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
        phenotype = pheno_dir.name
        for fname, db in RESOURCE_FILES.items():
            f = pheno_dir / fname
            if not f.exists():
                continue
            df = pd.read_csv(f, sep="\t")
            df["phenotype"] = phenotype
            df["database"] = db
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def clean_common(df):
    df = df.copy()
    for col in ["Old P-value", "Old Adjusted P-value"]:
        if col in df.columns:
            df = df.drop(columns=[col])
    if "P-value" in df.columns:
        df["P-value"] = pd.to_numeric(df["P-value"], errors="coerce")
    if "Adjusted P-value" in df.columns:
        df["Adjusted P-value"] = pd.to_numeric(df["Adjusted P-value"], errors="coerce")
    if "Combined Score" in df.columns:
        df["Combined Score"] = pd.to_numeric(df["Combined Score"], errors="coerce")
    return df

def filter_like_old_code(all_df):
    df = clean_common(all_df)

    out = {}
    for db in ["ENCODE", "GO_MF", "Reactome", "GWAS_Cat"]:
        out[db] = df[df["database"] == db].copy()

    for db in ["ENCODE", "GO_MF", "Reactome"]:
        d = out[db]
        if d.empty or not {"Genes", "P-value", "Term", "phenotype"}.issubset(d.columns):
            out[db] = d
            continue

        d["Genes"] = d["Genes"].astype(str)
        d = d[d["Genes"].str.contains(";", na=False) & (d["P-value"] <= 0.05)].copy()

        # genes > 3
        d = d[d["Genes"].apply(lambda g: len([x for x in g.split(";") if x.strip()]) > 3)].copy()

        if d.empty:
            out[db] = d
            continue

        min_rows = (
            d.groupby("phenotype", as_index=False)
             .apply(lambda grp: grp.loc[grp["P-value"].idxmin()])
             .reset_index(drop=True)
        )
        keep_terms = set(min_rows["Term"].unique())
        out[db] = d[d["Term"].isin(keep_terms)].copy()

        
    g = out["GWAS_Cat"]
    if (not g.empty) and {"Term", "P-value", "Adjusted P-value", "phenotype"}.issubset(g.columns):
        g = g[g["Adjusted P-value"] < 0.05].copy()
        g = g[g["Term"].astype(str).str.contains(KEYWORDS_RE, na=False)].copy()
    
        def pick_best_term(df, pheno, used_terms):
            sub = df[df["phenotype"] == pheno].sort_values("P-value")
            if sub.empty:
                return None

            for t in sub["Term"].astype(str):
                if t not in used_terms:
                    return t
            
            return str(sub.iloc[0]["Term"])
    
        used_terms = set()
    
        target_phenos = []
        for trait in ("SBP", "DBP"):
            pooled = f"{trait}_pre_ADJ"
            pre1   = f"{trait}_pre_1_ADJ"
            pre2   = f"{trait}_pre_2_ADJ"
            second = pre2 if not g[g["phenotype"] == pre2].empty else pre1
            target_phenos.extend([pooled, second])
    
        for ph in target_phenos:
            t = pick_best_term(g, ph, used_terms)
            if t is not None:
                used_terms.add(t)
    
        TARGET_N_TERMS = 8
        if len(used_terms) < TARGET_N_TERMS:
            for t in g.sort_values("P-value")["Term"].astype(str).unique():
                if t not in used_terms:
                    used_terms.add(t)
                    if len(used_terms) >= TARGET_N_TERMS:
                        break
    
        if used_terms:
            g = g[g["Term"].astype(str).isin(used_terms)].copy()
    
    out["GWAS_Cat"] = g

    combined = pd.concat(list(out.values()), ignore_index=True)
    return combined

def balloon_plot(df, out_svg="<PATH_TO_BALLOON_PLOT_SVG>"):
    need = {"phenotype", "database", "Term", "P-value", "Combined Score"}
    df = df.copy()
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df["P-value"] = pd.to_numeric(df["P-value"], errors="coerce")
    df["Combined Score"] = pd.to_numeric(df["Combined Score"], errors="coerce")
    df = df.dropna(subset=["P-value", "Combined Score", "phenotype", "database", "Term"]).copy()
    df = df[df["P-value"] > 0].copy()

    df["Term_db"] = "[" + df["database"].astype(str) + "] " + df["Term"].astype(str)

    phenos_in_data = [p for p in DESIRED_ORDER if p in set(df["phenotype"])]

    terms = df["Term_db"].drop_duplicates().tolist()

    term2idx = {t: i for i, t in enumerate(terms)}
    pheno2idx = {p: i for i, p in enumerate(phenos_in_data)}

    x_vals, y_vals, neglog10_p, scores = [], [], [], []
    for _, r in df.iterrows():
        p = r["phenotype"]
        t = r["Term_db"]
        if p not in pheno2idx:
            continue
        x_vals.append(pheno2idx[p])
        y_vals.append(term2idx[t])
        neglog10_p.append(-np.log10(float(r["P-value"])))
        scores.append(float(r["Combined Score"]))

    plt.rcParams.update({"font.size": 14})
    fig, ax = plt.subplots(figsize=(20, 10))

    size_factor = 20 
    sc = ax.scatter(
        x_vals,
        y_vals,
        s=[s * size_factor for s in scores],
        c=neglog10_p,
        cmap="viridis",
        alpha=0.8,
        edgecolors="k",
    )

    ax.set_xticks(range(len(phenos_in_data)))
    ax.set_xticklabels(phenos_in_data, rotation=45, ha="left", fontsize=14)
    ax.set_yticks(range(len(terms)))
    ax.set_yticklabels(terms, fontsize=14)

    ax.xaxis.tick_top()
    ax.tick_params(axis="x", labelbottom=False, labeltop=True)

    for i in range(len(terms)):
        ax.axhline(i, linestyle="--", color="lightgray", linewidth=0.5)
    for j in range(len(phenos_in_data)):
        ax.axvline(j, linestyle="--", color="lightgray", linewidth=0.5)

    cbar = plt.colorbar(sc, ax=ax, pad=0.15)
    cbar.set_label("-log10(P-value)", fontsize=14)
    cbar.ax.tick_params(labelsize=14)

    scores_min, scores_max = min(scores), max(scores)
    n_bubbles = 5
    values = np.linspace(scores_min, scores_max, n_bubbles)
    for val in values:
        ax.scatter([], [], s=val * size_factor, c="gray", alpha=0.8, edgecolors="k", label=f"{val:.2f}")

    ax.legend(
        title="Combined Score",
        labelspacing=1.5,
        borderpad=1,
        loc="center right",
        bbox_to_anchor=(1.2, 0.5),
        frameon=False,
        fontsize=14,
        title_fontsize=14,
    )

    plt.tight_layout()
    plt.savefig(out_svg, format="svg")
    plt.show()

In [ ]:
all_df = load_all_tables(ROOT)
combined_df = filter_like_old_code(all_df)
balloon_plot(combined_df, out_svg="<PATH_TO_BALLOON_PLOT_SVG>")

In [ ]:
# --- Supplementary enrichment tables ---

PADJ_THRESHOLD = 0.05
OUTDIR = ROOT

all_df = load_all_tables(ROOT)
df = clean_common(all_df)

# keep only adjusted p-value significant rows
df = df[df["P-value"] <= PADJ_THRESHOLD].copy()

# (1) GWAS Catalog table
gwas_table = (
    df[df["database"] == "GWAS_Cat"]
    .sort_values(["phenotype", "Adjusted P-value", "P-value", "Term"])
)

# (2) Other <PATH_TO_ENRICHMENT_DIR> databases
other_table = (
    df[df["database"] != "GWAS_Cat"]
    .sort_values(["database", "phenotype", "Adjusted P-value", "P-value", "Term"])
)

# write tables
gwas_out = OUTDIR / "<PATH_TO_SUPPLEMENTARY_TABLE_GWAS_CATALOG_TSV>"
other_out = OUTDIR / "<PATH_TO_SUPPLEMENTARY_TABLE_OTHER_DATABASES_TSV>"

gwas_table.to_csv(gwas_out, sep="\t", index=False)
other_table.to_csv(other_out, sep="\t", index=False)

print(f"GWAS catalog table written to: {gwas_out}")
print(f"Other databases table written to: {other_out}")

print("\nCounts:")
print("GWAS catalog rows:", len(gwas_table))
print("Other database rows:", len(other_table))